# Configurações

## Pacotes

### Instalação

In [ ]:
!pip install geopandas

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 40.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 67.0 MB/s eta 0:00:00


In [ ]:
!pip install rasterstats

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 53.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 54.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.9/137.9 kB 17.8 MB/s eta 0:00:00
  Attempting uninstall: fiona
    Found existing installation: Fiona 1.9.4
    Uninstalling Fiona-1.9.4:
      Successfully uninstalled Fiona-1.9.4


### Chamar

In [ ]:
from google.colab import drive
from google.colab import files

import pandas as pd
import geopandas
from shapely import wkt

from rasterstats import  point_query

import copy

## Conexao com a pasta

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/My Drive/PDPA

/content/drive/My Drive/PDPA


## Funções

In [ ]:
# Criação de variáveis dicotômicas
def flg_func(value):
    if pd.isna(value):
        return 0
    else:
      return 1

#Arquivos

In [ ]:
df_coordenadas_mapeadas = pd.read_csv('coordenadas_mapeadas.csv'
                     ,sep = ','
                     ,decimal='.'
                      )

df_coordenadas_mapeadas['geometry'] = df_coordenadas_mapeadas['geometry'].apply(wkt.loads)
gdf_coordenadas_mapeadas = geopandas.GeoDataFrame(df_coordenadas_mapeadas, crs='epsg:4326')

# Seleção das variáveis geomorfométricas

In [ ]:
lista_arquivos_tif = ['22S435ZN.tif',
                      '22S435SN.tif',
                      '22S435ON.tif',
                      '22S435VN.tif',
                      '22S435HN.tif',
                      '22S435SA.tif',
                      '22S435SB.tif',
                      '22S435SC.tif',
                      '22S435OC.tif',
                      '22S435V3.tif',
                      '22S435V5.tif',
                      '22S435H3.tif',
                      '22S435H5.tif',
                      '22S435FT.tif',
                      '22S435RS.tif',
                      '22S435DD.tif']

lista_nome_variaveis = ['Altitude_numerica',
                      'Declividade_numerica',
                      'Orientacao_numerica',
                      'Curv_Vertical_numerica',
                      'Curv_Horizontal_numerica',
                      'Declividade_classes_A',
                      'Declividade_classes_B',
                      'Declividade_classes_C',
                      'Orientacao_octantes',
                      'Curv_Vertical_3classes',
                      'Curv_Vertical_5classes',
                      'Curv_Horizontal_3classes',
                      'Curv_Horizontal_5classes',
                      'Forma_de_terreno_classes',
                      'Relevo_sombreado_numerico',
                      'ADD_divisores_talvegues']

In [ ]:
for variavel in range(len(lista_nome_variaveis)):
  df_coordenadas_mapeadas.loc[:,lista_nome_variaveis[variavel]]= point_query(df_coordenadas_mapeadas.loc[:,'geometry'], lista_arquivos_tif[variavel])

Streaming output truncated to the last 5000 lines.


In [ ]:
df_coordenadas_mapeadas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3135 entries, 0 to 3134
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   geometry                   3135 non-null   object 
 1   n_coord                    3135 non-null   int64  
 2   Altitude_numerica          3135 non-null   float64
 3   Declividade_numerica       3135 non-null   float64
 4   Orientacao_numerica        3135 non-null   float64
 5   Curv_Vertical_numerica     3135 non-null   float64
 6   Curv_Horizontal_numerica   3135 non-null   float64
 7   Declividade_classes_A      3135 non-null   float64
 8   Declividade_classes_B      3135 non-null   float64
 9   Declividade_classes_C      3135 non-null   float64
 10  Orientacao_octantes        3135 non-null   float64
 11  Curv_Vertical_3classes     3135 non-null   float64
 12  Curv_Vertical_5classes     3135 non-null   float64
 13  Curv_Horizontal_3classes   3135 non-null   float

In [ ]:
#df_coordenadas_mapeadas['geometry'] = df_coordenadas_mapeadas['geometry'].apply(wkt.loads)
gdf_relevo = geopandas.GeoDataFrame(df_coordenadas_mapeadas, crs='epsg:4326')

# Informação de vegetação

In [ ]:
gdf_vegetacao = geopandas.read_file("vege_area_mu_3303302.shp").to_crs(4326)

In [ ]:
gdf_relevo_veg = geopandas.tools.sjoin(gdf_relevo, gdf_vegetacao[['leg_sup','legenda_1','legenda_2','geometry']], predicate="within", how='left')

In [ ]:
_dummies = pd.get_dummies(gdf_relevo_veg, columns = None, drop_first = False)

In [ ]:
_dummies = _dummies.rename(columns={"	leg_sup_Massa D´água": "Massa_Dagua"
                            , "leg_sup_Vegetação Natural Dominante": "Vegetacao_Natural_Dominante"
                            , "leg_sup_Área Antrópica Dominante": "Area_Antropica_Dominante"
                            , "legenda_1_Floresta Ombrófila Densa": "Floresta_Ombrofila_Densa"
                            , "legenda_1_Formação Pioneira": "Formacao_Pioneira"
                            , "legenda_2_Floresta Ombrófila Densa Submontana": "Floresta_Ombrofila_Densa_Submontana"
                            , "legenda_2_Formação Pioneira com influência fluviomarinha": "Formacao_Pioneira_com_influencia_fluviomarinha"
                            , "legenda_2_Influência urbana": "Influencia_urbana"
                            , "legenda_2_Vegetação Secundária": "Vegetacao_Secundaria"}).drop(["index_right", "legenda_2_Indiscriminada"], axis=1)

In [ ]:
gdf_relevo = copy.deepcopy(_dummies)

In [ ]:
del(gdf_relevo_veg); del(gdf_vegetacao); del(_dummies)

# Informação de Pedologia

In [ ]:
gdf_pedologia = geopandas.read_file("pedo_area_mu_3303302.shp").to_crs(4326)

In [ ]:
gdf_relevo_ped = geopandas.tools.sjoin(gdf_relevo, gdf_pedologia[['leg_ordem','legenda_2','geometry']], predicate="within", how='left')

In [ ]:
_dummies = pd.get_dummies(gdf_relevo_ped, columns = None, drop_first = False)

In [ ]:
_dummies = _dummies.rename(columns={"leg_ordem_ARGISSOLO": "Argilossolo"
                            , "leg_ordem_GLEISSOLO": "Gleissolo"
                            , "legenda_2_ARGISSOLO AMARELO": "Argilossolo_Amarelo"
                            , "legenda_2_ARGISSOLO VERMELHO-AMARELO": "Argilossolo_Vermelho_Amarelo"
                            , "legenda_2_GLEISSOLO MELÂNICO": "Gleissolo_Melanico"
                            , "legenda_2_ÁREA URBANA": "Area_Urbana"}).drop(["index_right","leg_ordem_OUTROS"], axis=1)

In [ ]:
gdf_relevo = copy.deepcopy(_dummies)

In [ ]:
del(gdf_relevo_ped); del(gdf_pedologia); del(_dummies)

# Unidade de Conservação

In [ ]:
gdf_u_conservacao = geopandas.read_file('Unidades_de_Conservacao_Integral.geojson')

In [ ]:
DR = gdf_u_conservacao.iloc[[3]]
ST = gdf_u_conservacao.iloc[[4]]
res_difference = DR.overlay(ST, how='difference')


In [ ]:
gdf_u_conservacao = pd.concat([gdf_u_conservacao, res_difference])
gdf_u_conservacao = gdf_u_conservacao.drop(3, axis=0)

In [ ]:
gdf_relevo_conservacao = geopandas.tools.sjoin(gdf_relevo, gdf_u_conservacao[['tx_tipounidade','tx_planmanejo','geometry']], predicate="within", how='left').drop_duplicates()

In [ ]:
_dummies = pd.get_dummies(gdf_relevo_conservacao, columns = None, drop_first = False)

In [ ]:
_dummies = _dummies.rename(columns={"tx_tipounidade_Proteção Integral": "Unidades_de_Conservacao_Protecao_Integral"
                            , "tx_tipounidade_ ": "Unidades_de_Conservacao_Protecao_N_Integral"
                            , "tx_planmanejo_SIM": "Plano_de_Manejo"
                            , "tx_planmanejo_NÃO": "Sem_Plano_de_Menejo"}).drop(["index_right"], axis=1).drop_duplicates()

In [ ]:
gdf_relevo = copy.deepcopy(_dummies)

In [ ]:
del(gdf_u_conservacao); del(DR); del(ST); del(res_difference); del(gdf_relevo_conservacao); del(_dummies)

# Informação de Comunidades

In [ ]:
gdf_comunidades = geopandas.read_file('Comunidades.geojson')

In [ ]:
gdf_relevo_comunidades = geopandas.tools.sjoin(gdf_relevo, gdf_comunidades, predicate="within", how='left')

In [ ]:
gdf_relevo_comunidades['flg_comunidades'] = gdf_relevo_comunidades['OBJECTID'].map(flg_func)

In [ ]:
gdf_relevo_comunidades = gdf_relevo_comunidades.drop(columns=['index_right','OBJECTID','Nome','Area_ha'])

In [ ]:
del(gdf_comunidades)

# Informações de uso de solo

In [ ]:
gdf = gdf_relevo_comunidades

In [ ]:
variaveis_us = ['agricola'
                ,'exploracao_mineral'
                ,'corpo_hidrico'
                ,'rocha'
                ,'cobertura_vegetal'
                ,'afloramento_rochoso'
                ,'favela'
                ,'ocupacao_desordenada']

In [ ]:
for variavel in variaveis_us:

  gdf_uso_solo = geopandas.read_file('Uso_do_solo_' + variavel + '.geojson')

  gdf = geopandas.tools.sjoin(gdf, gdf_uso_solo, predicate="within", how='left')

  gdf['flg_'+ variavel] = gdf['OBJECTID_1'].map(flg_func)

  gdf = gdf.drop(columns=['index_right','OBJECTID_1','Nome','Código','Área','comparacao','Uso_do_Sol','Nome_1','OBS_1','OBS_12','Observacao','OBS','Observ']).drop_duplicates()


In [ ]:
del(gdf_uso_solo)

# Informação de área de risco

In [ ]:
gdf_risco = geopandas.read_file('areas_de_risco_Defesa_Civil.geojson')

gdf = geopandas.tools.sjoin(gdf, gdf_risco, predicate="within", how='left')

gdf['flg_areas_de_risco'] = gdf['OBJECTID'].map(flg_func)

gdf = gdf.drop(columns=['index_right','OBJECTID','tx_localid'])

In [ ]:
gdf['graurisc'] = gdf['graurisc'].transform(lambda x: x.fillna(0))

In [ ]:
gdf = pd.get_dummies(gdf,prefix_sep='_', columns=['Acoes_DC'])

# Especificar estação para coordenadas

In [ ]:
df_voronoi = pd.read_csv('voronoi_prefeitura.csv'
                     ,sep = ','
                     ,decimal='.'
                      )

In [ ]:
df_voronoi['geometry'] = df_voronoi['geom'].apply(wkt.loads)
gdf_voronoi = geopandas.GeoDataFrame(df_voronoi, crs='epsg:4326').drop_duplicates() #id 521, que é o Piratininga 2

In [ ]:
pointInPolys = geopandas.tools.sjoin(gdf, gdf_voronoi, predicate="within", how='left')

In [ ]:
pointInPolys['lat_ocr'] = pointInPolys.geometry.y
pointInPolys['lon_ocr'] = pointInPolys.geometry.x

# Salvar arquivo
Coordenadas e variáveis

In [ ]:
pointInPolys.to_csv('coordenadas_mapeadas_variaveis.csv',index=False, encoding = 'utf-8-sig')
files.download('coordenadas_mapeadas_variaveis.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_pointInPolys = pd.read_csv('coordenadas_mapeadas_variaveis.csv'
                     ,sep = ','
                     ,decimal='.'
                      )

df_pointInPolys['geometry'] = df_pointInPolys['geometry'].apply(wkt.loads)
pointInPolys = geopandas.GeoDataFrame(df_pointInPolys, crs='epsg:4326')

In [ ]:
pointInPolys['lat_ocr'] = pointInPolys.geometry.y
pointInPolys['lon_ocr'] = pointInPolys.geometry.x

# Espaços no tempo

O menor horário com ocorrência é id_tempo 117049888, analisar chuva diária (lag 24*60)

In [ ]:
lag = 24*60
#11615581 - 13758301

In [ ]:
lista_tempo = list(range(11615581,  13758301, lag ))

# Informações de chuva

In [ ]:
df_chuva = pd.read_csv('Tempo_estacao_24h.csv'
                     ,sep = ','
                     ,decimal='.'
                     ,encoding = 'utf-8'
                      )

In [ ]:
df_ocorr = pd.read_csv('amostra_aleatoria_coord_100m.csv'
                     ,sep = ','
                     ,decimal='.'
                     ,encoding = 'utf-8'
                      )

In [ ]:
df_ocorr

,id_tempo,n_coord
0,11049888,2666.0
1,11049661,2667.0
2,11053981,1732.0
3,11053981,2668.0
4,11055661,2669.0
...,...,...
1094,12602072,2722.0
1095,12602158,3124.0
1096,12607928,2924.0
1097,12609378,2752.0


In [ ]:
for i in range(len(df_ocorr['id_tempo'])):

  sla = (((df_chuva.loc[:,'id_tempo' ]- 24 * 60) < (df_ocorr.loc[i,'id_tempo'] )) &  ((df_chuva.loc[:,'id_tempo' ]) >= (df_ocorr.loc[i,'id_tempo'] )))

In [ ]:
sum(df_ocorr['id_tempo'][1] > df_chuva['id_tempo']- 24 * 60)

0

In [ ]:
df_ocorr['id_tempo']

0       11049888
1       11049661
2       11053981
3       11053981
4       11055661
          ...   
1094    12602072
1095    12602158
1096    12607928
1097    12609378
1098    12609235
Name: id_tempo, Length: 1099, dtype: int64

In [ ]:
df_chuva['id_tempo']

0        11615581
1        11615581
2        11615581
3        11615581
4        11615581
           ...   
30336    13228381
30337    13228381
30338    13228381
30339    13228381
30340    13228381
Name: id_tempo, Length: 30341, dtype: int64

In [ ]:
df_chuva = df_chuva.merge( pointInPolys.drop(columns=['Acoes_DC_0', 'flg_corpo_hidrico',	'Acoes_DC_1',	'Acoes_DC_2',	'Acoes_DC_3','index_right','lat','long','geom','Sem_Plano_de_Menejo']),
                                   right_on=['id_estacao'],
                                   left_on=['id_estacao'],
                                   how='inner')

In [ ]:
df_chuva['n_coord'] = df_chuva['n_coord'].astype('int')
df_chuva['id_tempo'] = df_chuva['id_tempo'].astype('int')

df_ocorr['n_coord'] = df_ocorr['n_coord'].astype('int')
df_ocorr['id_tempo'] =df_ocorr['id_tempo'].astype('int')

In [ ]:
df_chuva = df_chuva.merge( df_ocorr,
                                   right_on=['n_coord'],
                                   left_on=['n_coord'],
                                   how='left')

In [ ]:
df_chuva['ocorrencia'] = 0
_ocorrencia = (((df_chuva.loc[:,'id_tempo_x' ]- 24 * 60) < (df_chuva.loc[:,'id_tempo_y'] )) &  ((df_chuva.loc[:,'id_tempo_x' ]) >= (df_chuva.loc[:,'id_tempo_y'] )))
df_chuva.loc[_ocorrencia,'ocorrencia'] = 1

In [ ]:
df_chuva = df_chuva.drop(columns=['id_tempo_y']).drop_duplicates().sort_values(by=['id_tempo_x', 'n_coord', 'ocorrencia'])

In [ ]:
#Para remover os duplicados de n_coord e time, onde 1 recebe 0 e outro 1
df_chuva = df_chuva.drop_duplicates(subset=['id_tempo_x', 'n_coord'], keep='last')

# DataFrame final

In [ ]:
df_final = df_chuva[['n_coord'
,'id_estacao'
,'ocorrencia'
,'lon_ocr'
,'lat_ocr'
,'id_tempo_x'
,'tempo'
 ,'vintequatrohoras'
 ,'Altitude_numerica'
 ,'Declividade_numerica'
 ,'Orientacao_numerica'
 ,'Curv_Vertical_numerica'
 ,'Curv_Horizontal_numerica'
 ,'Declividade_classes_A'
 ,'Declividade_classes_B'
 ,'Declividade_classes_C'
 ,'Orientacao_octantes'
 ,'Curv_Vertical_3classes'
 ,'Curv_Vertical_5classes'
 ,'Curv_Horizontal_3classes'
 ,'Curv_Horizontal_5classes'
 ,'Forma_de_terreno_classes'
 ,'Relevo_sombreado_numerico'
 ,'ADD_divisores_talvegues'
 ,'Vegetacao_Natural_Dominante'
,'Area_Antropica_Dominante'
,'legenda_2_Pecuária (pastagens)'
,'Floresta_Ombrofila_Densa'
,'Formacao_Pioneira'
,'Floresta_Ombrofila_Densa_Submontana'
,'Influencia_urbana'
,'Vegetacao_Secundaria'
,'Argilossolo'
,'Gleissolo'
,'Argilossolo_Vermelho_Amarelo'
,'Gleissolo_Melanico'
,'Area_Urbana'
,'Unidades_de_Conservacao_Protecao_N_Integral'
,'Unidades_de_Conservacao_Protecao_Integral'
,'Plano_de_Manejo'
 ,'flg_comunidades'
 ,'flg_agricola'
 ,'flg_exploracao_mineral'
 ,'flg_rocha'
 ,'flg_cobertura_vegetal'
 ,'flg_afloramento_rochoso'
 ,'flg_favela'
 ,'flg_ocupacao_desordenada'
 ,'flg_areas_de_risco'
 ,'graurisc']]

In [ ]:
df_final.to_csv('df_final_4.csv',index=False, encoding = 'utf-8-sig')
files.download('df_final_4.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>